In [24]:
from __future__ import annotations
import asyncio
from typing import Optional, List
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi
import math
import pandas as pd
from runtime_support import (
    setup_client_from_env,
    api_navigate_ship,
    api_get_ship_nav,
    api_purchase_cargo,
    build_fleet_object
    )
from core_helpers import (
    init_world_state
)

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)

#Build all in-memory objects once (fleet_activity_obj, waypoints_ref_obj, waypoint_traits_obj, HQ)
world_state = await init_world_state(fleet_api, agents_api, systems_api)

In [ ]:
from adapters.ships_activity_adapter import merge_activity_with_nav
from adapters.ships_specs_adapter import adapt_ships_specs_from_ship
from core_helpers import adapt_ships_activity_from_ship, upsert_many
from typing import Any, Dict, Iterable, List, Optional, Tuple
from runtime_support import call_sdk, unwrap_data
from dataclasses import dataclass, field
from domain.ships_activity import ShipsActivity
from domain.ships_specs import ShipsSpecs
from domain.ship_market import ShipMarketRow
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi

with setup_client_from_env() as client:
        fleet_api = FleetApi(client)
        agents_api = AgentsApi(client)
        systems_api = SystemsApi(client)

@dataclass
class FleetState:
    """Local, easily-referenced state for your session."""
    # Activity & Specs keyed by ship symbol
    activities: Dict[str, ShipsActivity] = field(default_factory=dict)
    specs: Dict[str, ShipsSpecs] = field(default_factory=dict)

    # Shipyard listings cached by waypoint
    ship_market: Dict[str, List[ShipMarketRow]] = field(default_factory=dict)

    def ensure_activity(self, symbol: str) -> ShipsActivity:
        a = self.activities.get(symbol)
        if a is None:
            a = ShipsActivity(symbol=symbol)
            self.activities[symbol] = a
        return a

    def update_activity_from_nav(self, symbol: str, nav_dto: Any) -> ShipsActivity:
        current = self.ensure_activity(symbol)
        updated = merge_activity_with_nav(current, nav_dto)
        self.activities[symbol] = updated
        return updated

    def update_activity_from_refuel(self, symbol: str, resp_dto: Any) -> ShipsActivity:
        """Use refuel response to update fuel state in local activity."""
        current = self.ensure_activity(symbol)
        fuel_cur = getattr(getattr(resp_dto, "fuel", None), "current", None)
        fuel_cap = getattr(getattr(resp_dto, "fuel", None), "capacity", None)
        patched = current.model_copy(update={
            "fuel_current": fuel_cur if fuel_cur is not None else current.fuel_current,
            "fuel_capacity": fuel_cap if fuel_cap is not None else current.fuel_capacity,
        })
        self.activities[symbol] = patched
        return patched

async def local_fleet_state(fleet: FleetApi) -> FleetState:
    """Load ships once, build local state (activity + specs) and persist to DB."""
    resp = await call_sdk(fleet, "get_my_ships")
    ships: Iterable[Any] = unwrap_data(resp)
    ships = list(ships)
    if not ships:
        raise SystemExit("[FATAL] No ships returned; check token/agent.")

    activities = [adapt_ships_activity_from_ship(d) for d in ships]
    specs = [adapt_ships_specs_from_ship(d) for d in ships]
    state = FleetState(
        activities={a.symbol: a for a in activities if a.symbol},
        specs={s.symbol: s for s in specs if s.symbol},
    )
    return state

# --------- define ship roles -----------
state = await local_fleet_state(fleet_api)
print(state.specs)

# Extract symbol for a given role
def get_symbol_by_role(ships_dict, role):
    for ship in ships_dict.values():
        if ship.role == role:
            return ship.symbol
    return None

command_ship = get_symbol_by_role(state.specs, "COMMAND")
sattelite = get_symbol_by_role(state.specs, "SATTELITE")
print(command_ship)

{'KIJINIBIBI-1': ShipsSpecs(symbol='KIJINIBIBI-1', role='COMMAND', frame_name='Frigate', frame_module_slots=8, frame_mounting_points=5, engine_name='Ion Drive II', speed=36, mounts=['MOUNT_SENSOR_ARRAY_II', 'MOUNT_GAS_SIPHON_II', 'MOUNT_MINING_LASER_II', 'MOUNT_SURVEYOR_II'], modules=['MODULE_CARGO_HOLD_II', 'MODULE_CREW_QUARTERS_I', 'MODULE_CREW_QUARTERS_I', 'MODULE_MINERAL_PROCESSOR_I', 'MODULE_GAS_PROCESSOR_I'], capacity=40), 'KIJINIBIBI-2': ShipsSpecs(symbol='KIJINIBIBI-2', role='SATELLITE', frame_name='Probe', frame_module_slots=0, frame_mounting_points=0, engine_name='Impulse Drive I', speed=9, mounts=[], modules=[], capacity=0)}
KIJINIBIBI-1


In [25]:
# ---------- get market waypoints from world state ----------
traits = world_state.traits.by_wp
print(f"[BOOT] fleet={len(state.fleet.by_symbol)} ships, waypoints={len(state.waypoints.by_symbol)} (system), traits={sum(len(v) for v in state.traits.by_wp.values())}")
async def get_closest_wp_by_trait(trait: str):

    state = await init_world_state(fleet_api, agents_api, systems_api)
    traits = state.traits.by_wp
    data = []
    for wp_symbol, rows_list in traits.items():
        for row in rows_list:
            if row.trait_symbol == "MARKETPLACE":
                x = row.x or 0
                y = row.y or 0
                data.append({"waypoint": wp_symbol, "x": x, "y": y, "distance": math.hypot(x, y)})
    data = pd.DataFrame(data).sort_values("distance", ascending=True).head(50).reset_index(drop=True)
    #print (data.to_string(index=False))
    return data

markets_df = await get_closest_wp_by_trait("MARKETPLACE")
#print(markets_df)
#print(markets_df.to_string(index=False))

[BOOT] fleet=2 ships, waypoints=91 (system), traits=267


In [26]:
# ---------- call market waypoints in a logical order ----------

import math
from typing import List, Optional, Tuple
import numpy as np
import pandas as pd


def _euclid(a: Tuple[float, float], b: Tuple[float, float]) -> float:
    dx, dy = a[0] - b[0], a[1] - b[1]
    return math.hypot(dx, dy)


def _pairwise_dist_matrix(coords: np.ndarray) -> np.ndarray:
    # coords: (N,2)
    # returns (N,N) symmetric distance matrix
    diff = coords[:, None, :] - coords[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=2))


def _nearest_neighbor_tour(D: np.ndarray, start_idx: int) -> List[int]:
    n = D.shape[0]
    unvisited = set(range(n))
    tour = [start_idx]
    unvisited.remove(start_idx)
    cur = start_idx
    while unvisited:
        nxt = min(unvisited, key=lambda j: D[cur, j])
        tour.append(nxt)
        unvisited.remove(nxt)
        cur = nxt
    return tour


def _two_opt_once(tour: List[int], D: np.ndarray) -> Tuple[List[int], bool]:
    """Perform a single improvement pass; returns (new_tour, improved?)."""
    n = len(tour)
    best = tour
    improved = False

    def seg_len(i1, i2):
        a, b = best[i1], best[(i1 + 1) % n]
        c, d = best[i2], best[(i2 + 1) % n]
        return D[a, b] + D[c, d]

    for i in range(n - 2):
        for k in range(i + 2, n - (0 if i > 0 else 1)):
            before = seg_len(i, k)
            after = D[best[i], best[k]] + D[best[i + 1], best[(k + 1) % n]]
            if after + 1e-12 < before:
                best = best[: i + 1] + list(reversed(best[i + 1 : k + 1])) + best[k + 1 :]
                improved = True
    return best, improved


def _two_opt(tour: List[int], D: np.ndarray, max_iters: int = 50) -> List[int]:
    cur = tour
    for _ in range(max_iters):
        cur, improved = _two_opt_once(cur, D)
        if not improved:
            break
    return cur


def all_market_visitor(
    markets_df: pd.DataFrame,
    start_waypoint: Optional[str] = None,
    return_to_start: bool = False,
    improve_2opt: bool = True,
) -> pd.DataFrame:
    """
    Build a visit order that covers all markets with short total travel.

    Parameters
    ----------
    markets_df : DataFrame with columns ['waypoint', 'x', 'y']
    start_waypoint : optional waypoint symbol to start at.
                     If None, picks the market with smallest (x, then y).
    return_to_start : if True, closes the tour back to the start.
    improve_2opt : run a 2-opt local improvement after nearest-neighbor.

    Returns
    -------
    route_df : DataFrame with columns:
        - visit_idx (0..N-1 or 0..N if returning to start)
        - waypoint
        - x, y
        - leg_distance (0 for first row)
        - cumulative_distance
    """
    if markets_df.empty:
        return pd.DataFrame(columns=["visit_idx", "waypoint", "x", "y", "leg_distance", "cumulative_distance"])

    for col in ["waypoint", "x", "y"]:
        if col not in markets_df.columns:
            raise ValueError(f"markets_df must contain column '{col}'")

    df = markets_df[["waypoint", "x", "y"]].copy().reset_index(drop=True)
    coords = df[["x", "y"]].to_numpy(dtype=float)
    D = _pairwise_dist_matrix(coords)

    # Pick start index
    if start_waypoint is not None:
        if start_waypoint not in set(df["waypoint"]):
            raise ValueError(f"start_waypoint '{start_waypoint}' not found in markets_df['waypoint']")
        start_idx = int(df.index[df["waypoint"] == start_waypoint][0])
    else:
        # Default: left-most (min x), tie-breaker min y
        start_idx = int(df.sort_values(["x", "y"]).index[0])

    # Build initial tour and optionally improve
    tour = _nearest_neighbor_tour(D, start_idx)
    if improve_2opt and len(tour) >= 4:
        tour = _two_opt(tour, D)

    # Optionally close the tour by returning to start
    sequence = tour + ([tour[0]] if return_to_start else [])

    # Compute leg distances & cumulative
    legs = [0.0]
    cum = [0.0]
    for i in range(1, len(sequence)):
        a, b = sequence[i - 1], sequence[i]
        dist = D[a, b]
        legs.append(dist)
        cum.append(cum[-1] + dist)

    out = df.iloc[sequence].reset_index(drop=True)
    out.insert(0, "visit_idx", range(len(sequence)))
    out["leg_distance"] = np.round(legs, 3)
    out["cumulative_distance"] = np.round(cum, 3)
    return out

In [27]:
# -------- define route ----------

nav_resp = await api_get_ship_nav(fleet_api, command_ship)
cur_wp = nav_resp.waypoint_symbol

if __name__ == "__main__":
    route = all_market_visitor(
        markets_df=markets_df,
        start_waypoint=cur_wp,      # or e.g., "M00"
        return_to_start=True,    # set True to make a loop
        improve_2opt=True,
    )

    print(route)
    print("\nTotal distance:", route["cumulative_distance"].iloc[-1])

    visit_idx      waypoint    x    y  leg_distance  cumulative_distance
0           0   X1-SV25-H55   15  -43         0.000                0.000
1           1   X1-SV25-H53   15  -43         0.000                0.000
2           2   X1-SV25-H54   15  -43         0.000                0.000
3           3   X1-SV25-H52   15  -43         0.000                0.000
4           4   X1-SV25-D43  -16  -83        50.606               50.606
5           5   X1-SV25-D42  -16  -83         0.000               50.606
6           6    X1-SV25-A1  -24    8        91.351              141.957
7           7    X1-SV25-A2  -24    8         0.000              141.957
8           8    X1-SV25-A4  -24    8         0.000              141.957
9           9    X1-SV25-A3  -24    8         0.000              141.957
10         10   X1-SV25-F46  -70  -27        57.801              199.759
11         11   X1-SV25-F47  -70  -27         0.000              199.759
12         12   X1-SV25-F49  -70  -27         0.000

In [28]:
# ---------- init functions for writing to db and patrolling markets  ----------

from market_runtime import capture_market_for_waypoint
from db.auto_repo_sqlite import snapshot_many, TableSpec, upsert_many
from domain.market_rows import MarketGoodRow, MarketTransactionRow
import sqlite3

async def market_to_db(waypoint: str) -> None:
    # Fetch first (network I/O), then write to DB (short-lived connection)
    rows = await capture_market_for_waypoint(systems_api, waypoint)
    with sqlite3.connect("spacetraders.db") as conn:
        snapshot_many(conn, "market_goods", MarketGoodRow, rows["goods"])  # append-only history
        if rows["transactions"]:
            upsert_many(conn, TableSpec(table="market_transactions", pk="id"), rows["transactions"])


# --- main patrol loop ---
async def patrol_markets(ship_symbol: str, markets_df) -> None:
    # dedupe and coerce to plain list of strings
    waypoints: List[str] = list(dict.fromkeys(markets_df["waypoint"].astype(str).tolist()))
    print(waypoints)
    if not waypoints:
        print("[WARN] No waypoints to patrol.")
        return

    print(f"[PATROL] {ship_symbol} looping through {len(waypoints)} markets.")
    idx = 0
    while True:
        wp = waypoints[idx]
        try:
            print(f"[PATROL] -> Navigating to {wp} (#{idx+1}/{len(waypoints)})")
            nav_resp = await api_navigate_ship(fleet_api, ship_symbol, wp)
            print(f"[MARKET] Capturing {wp} …")
            await market_to_db(wp)
            await asyncio.sleep(1.0)  # small dwell

        except asyncio.CancelledError:
            raise
        except Exception as e:
            print(f"[ERR] Patrol step at {wp} failed: {e!r}")
            await asyncio.sleep(3.0)  # brief backoff

        # round-robin
        idx = (idx + 1) % len(waypoints)


In [ ]:
# ---------- run ----------
async def main():
    patroller = asyncio.create_task(patrol_markets(command_ship, route))
    await asyncio.gather(patroller)

await main()


['X1-SV25-H55', 'X1-SV25-H53', 'X1-SV25-H54', 'X1-SV25-H52', 'X1-SV25-D43', 'X1-SV25-D42', 'X1-SV25-A1', 'X1-SV25-A2', 'X1-SV25-A4', 'X1-SV25-A3', 'X1-SV25-F46', 'X1-SV25-F47', 'X1-SV25-F49', 'X1-SV25-B6', 'X1-SV25-B7', 'X1-SV25-J59', 'X1-SV25-J58', 'X1-SV25-I56', 'X1-SV25-I57', 'X1-SV25-K87', 'X1-SV25-K90', 'X1-SV25-K89', 'X1-SV25-E45', 'X1-SV25-E44', 'X1-SV25-G51', 'X1-SV25-C41', 'X1-SV25-C40', 'X1-SV25-CX5E']
[PATROL] KIJINIBIBI-1 looping through 28 markets.
[PATROL] -> Navigating to X1-SV25-H55 (#1/28)
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Seconds until arrival: 32.090823
Arrived and ready
[BOOT] Adapted 2 ships into fleet_object
After arriving, the status of KIJINIBIBI-1 is IN_ORBIT
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
KIJINIBIBI-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 68.574208
Arrived and ready
[MARKET] Capturing X1-SV25-H55 …
[PATR

In [ ]:
nav_resp = await api_get_ship_nav(fleet_api, "KIJINIBIBI-1")


system_symbol='X1-SV25' waypoint_symbol='X1-SV25-E44' route=ShipNavRoute(destination=ShipNavRouteWaypoint(symbol='X1-SV25-E44', type=<WaypointType.PLANET: 'PLANET'>, system_symbol='X1-SV25', x=14, y=52), origin=ShipNavRouteWaypoint(symbol='X1-SV25-H55', type=<WaypointType.MOON: 'MOON'>, system_symbol='X1-SV25', x=15, y=-43), departure_time=datetime.datetime(2025, 9, 15, 17, 56, 26, 70000, tzinfo=TzInfo(UTC)), arrival=datetime.datetime(2025, 9, 15, 17, 57, 47, 70000, tzinfo=TzInfo(UTC))) status=<ShipNavStatus.IN_ORBIT: 'IN_ORBIT'> flight_mode=<ShipNavFlightMode.CRUISE: 'CRUISE'>


In [14]:
print(nav_resp.waypoint_symbol)

X1-SV25-E44
